# MSigDB blitzgsea GSEA Analysis: saturation (random subsampling), CLAMPfull

**Environment:** `clamp-analyses` (Python kernel)

For CLAMPfull models in the random-subsampling saturation grid, this notebook:

1. Loads the Z matrix (gene loadings per LV) for each (rs_pct, K) combination, seed 1 only.
2. For each LV, runs `blitzgsea.gsea()` (ranked GSEA, full MSigDB v2026.1 library, `min_size=10`, `max_size=500`, `processes=4`, `shared_null=True`).
3. Keeps only **positive-NES** results (project convention — enrichment among high positive LV-loading genes only).
4. Takes the minimum `fdr` per pathway across LVs within each model.
5. **Capped K grid**: only K ∈ {86, 173, 432} per rs_pct level (1/5/10/25) — the full K grid (up to 1728) was measured at ~18s/LV and would take ~83h per model-type; capping keeps this to ~13.8h.
6. **Shared 100% anchor**: the true full-coverage model (K=1728, from `06_bp_coverage_rshall/06_bp_coverage_hall_rs_100`) is the same physical reference point for both saturation tracks (random and study) — computed once per model type and cached under `output/03_model_biology/00_archs4/_blitzgsea_100pct_anchor/CLAMPfull/`, reused by both `07_saturation_random` and `08_saturation_study` plot notebooks rather than recomputed per track.

No PDF export — this notebook only writes CSV result caches; plotting happens in the companion R notebook.

In [1]:
import time
import os
import pandas as pd
import blitzgsea as blitz

REPO_ROOT = "/home/msubirana/Documents/pivlab/clamp-analyses"
os.chdir(REPO_ROOT)

t_start = time.time()

## Paths

In [2]:
models_dir   = "output/01_model_building/04_archs4/07_saturation"
output_dir   = "output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull"
anchor_dir   = "output/03_model_biology/00_archs4/_blitzgsea_100pct_anchor/CLAMPfull"
anchor_z_path = "output/01_model_building/04_archs4/06_bp_coverage_rshall/06_bp_coverage_hall_rs_100/hall_coverage_rs100_seed_1/CLAMPfull_hall/Z.csv"

import os
os.makedirs(output_dir, exist_ok=True)
os.makedirs(anchor_dir, exist_ok=True)

## Capped grid spec

rs_pct levels 1/5/10/25, K capped to {86, 173, 432} (seed 1 only).

In [3]:
rs_pcts   = [1, 5, 10, 25]
k_values  = [86, 173, 432]
model_type = "CLAMPfull"
z_subdir   = "CLAMPfull_hall"
dir_pattern = "hall_saturation_rs{rs}_k{k}_seed_1"
seed = 1

## Load MSigDB gene sets as a blitzgsea library

In [4]:
def read_gmt(gmt_file):
    gene_sets = {}
    with open(gmt_file) as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            gene_sets[fields[0]] = fields[2:]
    return gene_sets

library = read_gmt("data/pathways/msigdb.v2026.1.Hs.symbols.gmt")
msigdb_genes = set(g for genes in library.values() for g in genes)
print(f"MSigDB gene sets loaded: {len(library)}")

MSigDB gene sets loaded: 35361


## Helper: run blitzgsea for one model

Returns a dict with `terms_fdr` (minimum FDR per MSigDB term across LVs, positive-NES only), plus metadata.

In [5]:
def run_blitzgsea_for_model(z_path, n_cores=4):
    Z = pd.read_csv(z_path, index_col=0)
    universe_genes = set(Z.index)
    term_overlap = {term: len(set(genes) & universe_genes) for term, genes in library.items()}
    n_total_msigdb = sum(1 for v in term_overlap.values() if v >= 10)

    per_lv = []
    for lv in Z.columns:
        lv_scores = Z[lv].dropna()
        signature = lv_scores.reset_index()
        signature.columns = [0, 1]
        signature = signature[signature[0].isin(msigdb_genes)]
        signature = signature.sort_values(1, ascending=False)
        if len(signature) == 0:
            continue
        result = blitz.gsea(signature, library, min_size=10, max_size=500,
                             processes=n_cores, shared_null=True)
        result = result[result["nes"] > 0]  # positive-side only (project convention)
        per_lv.append(result["fdr"])

    if len(per_lv) == 0:
        return None

    combined = pd.concat(per_lv, axis=1)
    terms_fdr = combined.min(axis=1, skipna=True)

    b_path = z_path.replace(z_subdir, "").rsplit("/", 1)[0] + "/B.csv" if False else \
             "/".join(z_path.split("/")[:-1]) + "/B.csv"
    try:
        n_samples = len(pd.read_csv(b_path, nrows=0).columns)
    except FileNotFoundError:
        n_samples = None

    return {
        "n_samples": n_samples,
        "n_lvs": Z.shape[1],
        "n_total_msigdb": n_total_msigdb,
        "terms_fdr": terms_fdr,
    }

## Shared 100% anchor (CLAMPfull)

Computed once, cached under `_blitzgsea_100pct_anchor/CLAMPfull/` — shared by both saturation tracks (random and study), since it's the same physical model regardless of which track's plot references it.

In [6]:
anchor_cache_path = os.path.join(anchor_dir, "anchor_msigdb_blitzgsea_gsea.csv")

if os.path.exists(anchor_cache_path):
    print(f"Loading cached 100% anchor: {anchor_cache_path}")
    anchor_df = pd.read_csv(anchor_cache_path)
else:
    print(f"Running blitzgsea on 100% anchor model ({model_type}, K=1728)...")
    t0 = time.time()
    res = run_blitzgsea_for_model(anchor_z_path)
    print(f"Anchor done in {time.time()-t0:.1f}s")
    if res is not None:
        anchor_df = pd.DataFrame({
            "term": res["terms_fdr"].index,
            "min_fdr": res["terms_fdr"].values,
        })
        anchor_df["n_samples"] = res["n_samples"]
        anchor_df["n_lvs"] = res["n_lvs"]
        anchor_df["n_total_msigdb"] = res["n_total_msigdb"]
        anchor_df.to_csv(anchor_cache_path, index=False)
        print(f"Saved: {anchor_cache_path}")
    else:
        anchor_df = None

Running blitzgsea on 100% anchor model (CLAMPfull, K=1728)...


Anchor done in 29978.3s
Saved: output/03_model_biology/00_archs4/_blitzgsea_100pct_anchor/CLAMPfull/anchor_msigdb_blitzgsea_gsea.csv


## Run capped grid: CLAMPfull

In [7]:
summary_rows = []

for rs in rs_pcts:
    for k in k_values:
        subdir = dir_pattern.format(rs=rs, k=k)
        z_path = os.path.join(models_dir, subdir, z_subdir, "Z.csv")
        cache_path = os.path.join(output_dir, f"rs{rs}_k{k}_seed{seed}_msigdb_blitzgsea_gsea.csv")

        if not os.path.exists(z_path):
            print(f"SKIP (no Z.csv): rs{rs} k{k}")
            continue

        if os.path.exists(cache_path):
            print(f"Loading cached: rs{rs} k{k}")
            cached = pd.read_csv(cache_path)
            n_samples = cached["n_samples"].iloc[0] if len(cached) else None
            n_lvs = cached["n_lvs"].iloc[0] if len(cached) else None
            n_total_msigdb = cached["n_total_msigdb"].iloc[0] if len(cached) else None
        else:
            print(f"Running blitzgsea: rs{rs} k{k} ({model_type})")
            t0 = time.time()
            res = run_blitzgsea_for_model(z_path)
            print(f"  done in {time.time()-t0:.1f}s")
            if res is None:
                continue
            df = pd.DataFrame({
                "term": res["terms_fdr"].index,
                "min_fdr": res["terms_fdr"].values,
            })
            df["n_samples"] = res["n_samples"]
            df["n_lvs"] = res["n_lvs"]
            df["n_total_msigdb"] = res["n_total_msigdb"]
            df.to_csv(cache_path, index=False)
            n_samples, n_lvs, n_total_msigdb = res["n_samples"], res["n_lvs"], res["n_total_msigdb"]
            print(f"  Saved: {cache_path}")

        summary_rows.append({
            "model_type": model_type, "rs_pct": rs, "k_val": k, "seed": seed,
            "n_samples": n_samples, "n_lvs": n_lvs, "n_total_msigdb": n_total_msigdb,
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df)

Running blitzgsea: rs1 k86 (CLAMPfull)


  done in 1675.2s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs1_k86_seed1_msigdb_blitzgsea_gsea.csv
Running blitzgsea: rs1 k173 (CLAMPfull)


  done in 3181.9s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs1_k173_seed1_msigdb_blitzgsea_gsea.csv
Running blitzgsea: rs1 k432 (CLAMPfull)


  done in 7753.5s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs1_k432_seed1_msigdb_blitzgsea_gsea.csv
Running blitzgsea: rs5 k86 (CLAMPfull)


  done in 1583.3s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs5_k86_seed1_msigdb_blitzgsea_gsea.csv
Running blitzgsea: rs5 k173 (CLAMPfull)


...

  done in 3188.8s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs10_k173_seed1_msigdb_blitzgsea_gsea.csv
Running blitzgsea: rs10 k432 (CLAMPfull)


  done in 7735.2s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs10_k432_seed1_msigdb_blitzgsea_gsea.csv
Running blitzgsea: rs25 k86 (CLAMPfull)


  done in 1690.7s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs25_k86_seed1_msigdb_blitzgsea_gsea.csv
Running blitzgsea: rs25 k173 (CLAMPfull)


  done in 3243.3s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs25_k173_seed1_msigdb_blitzgsea_gsea.csv
Running blitzgsea: rs25 k432 (CLAMPfull)


  done in 7693.0s
  Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/rs25_k432_seed1_msigdb_blitzgsea_gsea.csv
   model_type  rs_pct  k_val  seed  n_samples  n_lvs  n_total_msigdb
0   CLAMPfull       1     86     1       6057     86           28320
1   CLAMPfull       1    173     1       6057    173           28320
2   CLAMPfull       1    432     1       6057    432           28320
3   CLAMPfull       5     86     1      30282     86           28320
4   CLAMPfull       5    173     1      30282    173           28320
5   CLAMPfull       5    432     1      30282    432           28320
6   CLAMPfull      10     86     1      60562     86           28320
7   CLAMPfull      10    173     1      60562    173           28320
8   CLAMPfull      10    432     1      60562    432           28320
9   CLAMPfull      25     86     1     151405     86           28320
10  CLAMPfull      25    173     1     151405    173           28320
11  CLAMPfull      25  

In [8]:
summary_path = os.path.join(output_dir, "results_summary_msigdb_blitzgsea_gsea.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")
print(f"Total notebook time: {(time.time()-t_start)/3600:.2f}h")

Saved: output/03_model_biology/00_archs4/07_saturation_random/blitzgsea_gsea/CLAMPfull/results_summary_msigdb_blitzgsea_gsea.csv
Total notebook time: 22.27h
